# comfy_colab — SDXL images + Wan2.2 video on a free Colab T4

One self-contained notebook. **Runtime → Change runtime type → T4 GPU**, then run cells top-to-bottom once (setup + server), then re-run any **Generate** cell on demand to iterate.

| Cell | Run |
|------|-----|
| 1 Setup | once |
| 2 Video models | once (only if you want video) |
| 3 Start server | once |
| 4 Generate · SDXL image | on demand |
| 5 Generate · Wan2.2 video | on demand |
| 6 Display | after a generate cell |

Images/clips are saved to `ComfyUI/output/` on the VM (ephemeral) — download via the file browser, or use the Drive mount cell to persist.


In [ ]:
#@title 1 · Setup: ComfyUI + SDXL + GGUF node { display-mode: "form" }
import os, subprocess
COMFY = "/content/ComfyUI"
CKPT_DIR = f"{COMFY}/models/checkpoints"
CKPT = f"{CKPT_DIR}/sd_xl_base_1.0.safetensors"
SDXL_URL = ("https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/"
            "resolve/main/sd_xl_base_1.0.safetensors")

if not os.path.isdir(f"{COMFY}/.git"):
    print("Cloning ComfyUI...")
    subprocess.run(["git","clone","-q","https://github.com/comfyanonymous/ComfyUI.git", COMFY], check=True)
else:
    print("ComfyUI present")
print("Installing requirements...")
subprocess.run(["pip","install","-q","-r",f"{COMFY}/requirements.txt"], check=True)

# GGUF custom node (for Wan2.2 quantized model)
GGUF_NODE = f"{COMFY}/custom_nodes/ComfyUI-GGUF"
if not os.path.isdir(f"{GGUF_NODE}/.git"):
    print("Installing ComfyUI-GGUF node...")
    subprocess.run(["git","clone","-q","https://github.com/city96/ComfyUI-GGUF.git", GGUF_NODE], check=True)

if not os.path.exists(CKPT):
    print("Downloading SDXL base (~6.5 GB)...")
    os.makedirs(CKPT_DIR, exist_ok=True)
    subprocess.run(["wget","-q","--show-progress","-O",CKPT,SDXL_URL], check=True)
print(f"\nSetup done. SDXL: {os.path.getsize(CKPT)//1048576} MB")


In [ ]:
#@title 2 · (Optional) Download Wan2.2 5B video models { display-mode: "form" }
import os, subprocess
DIFF = f"{COMFY}/models/diffusion_models"
TE   = f"{COMFY}/models/text_encoders"
VAE  = f"{COMFY}/models/vae"
for d in (DIFF, TE, VAE): os.makedirs(d, exist_ok=True)

FILES = {
    # Wan2.2 TI2V 5B, GGUF Q4_K_M (~3.3 GB) — fits T4 VRAM comfortably
    f"{DIFF}/wan2.2_ti2v_5B-Q4_K_M.gguf":
        "https://huggingface.co/QuantStack/Wan2.2-TI2V-5B-GGUF/resolve/main/Wan2.2-TI2V-5B-Q4_K_M.gguf",
    # text encoder (umt5 fp8, ~6.4 GB) — freed after encode
    f"{TE}/umt5_xxl_fp8_e4m3fn_scaled.safetensors":
        "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    # Wan2.2 VAE (~1.3 GB)
    f"{VAE}/wan2.2_vae.safetensors":
        "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan2.2_vae.safetensors",
}
for path, url in FILES.items():
    if os.path.exists(path):
        print(f"  exists: {os.path.basename(path)} ({os.path.getsize(path)//1048576} MB)")
    else:
        print(f"  downloading {os.path.basename(path)} ...")
        subprocess.run(["wget","-q","--show-progress","-O",path,url], check=True)
print("\nVideo models ready.")


In [ ]:
#@title 3 · Start ComfyUI server { display-mode: "form" }
import subprocess, time, os, urllib.request
PORT = 8188
PIDFILE = "/content/comfy.pid"; LOGFILE = "/content/comfy.log"
def _alive(pid):
    try: os.kill(pid, 0); return True
    except Exception: return False
pid = None
if os.path.exists(PIDFILE):
    try: pid = int(open(PIDFILE).read().strip())
    except: pid = None
if pid and _alive(pid):
    print(f"server already running, pid={pid}")
else:
    print("launching server...")
    log = open(LOGFILE, "w")
    subprocess.Popen(["python","main.py","--listen","0.0.0.0","--port",str(PORT),
        "--output-directory", os.path.join(COMFY,"output")],
        cwd=COMFY, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    for i in range(120):
        time.sleep(1)
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=1)
            print(f"READY on http://127.0.0.1:{PORT}  (boot {i+1}s)"); break
        except Exception: pass
    else:
        print("TIMEOUT:\n"+open(LOGFILE).read()[-1500:])


In [ ]:
#@title 4 · Generate · SDXL image { display-mode: "form" }
prompt = "A tilted brass arrow mounted on a wooden base casts a crisp shadow onto a horizontal brass rail beneath it. Cobblestone workshop floor, copper pipes in background, soft window light from the left. Photorealistic steampunk still life, warm tones, detailed metal textures." #@param {type:"string"}
negative = "blurry, low quality, deformed, watermark, text" #@param {type:"string"}
seed = 7 #@param {type:"integer"}
import json, time, urllib.request, urllib.parse, os
SERVER = f"http://127.0.0.1:{PORT}"; OUT = f"{COMFY}/output"; os.makedirs(OUT, exist_ok=True)
wf = {"prompt":{
  "4":{"class_type":"CheckpointLoaderSimple","inputs":{"ckpt_name":"sd_xl_base_1.0.safetensors"}},
  "5":{"class_type":"EmptyLatentImage","inputs":{"width":1024,"height":1024,"batch_size":1}},
  "6":{"class_type":"CLIPTextEncode","inputs":{"text":prompt,"clip":["4",1]}},
  "7":{"class_type":"CLIPTextEncode","inputs":{"text":negative,"clip":["4",1]}},
  "3":{"class_type":"KSampler","inputs":{"seed":seed,"steps":25,"cfg":7.5,"sampler_name":"euler","scheduler":"normal","denoise":1.0,"model":["4",0],"positive":["6",0],"negative":["7",0],"latent_image":["5",0]}},
  "8":{"class_type":"VAEDecode","inputs":{"samples":["3",0],"vae":["4",2]}},
  "9":{"class_type":"SaveImage","inputs":{"filename_prefix":"sdxl","images":["8",0]}}}}
pid = json.loads(urllib.request.urlopen(urllib.request.Request(f"{SERVER}/prompt",
    data=json.dumps(wf).encode(), headers={"Content-Type":"application/json"})).read())["prompt_id"]
print("submitted:", pid)
t0=time.time(); LAST=None
while time.time()-t0<300:
    try: h=json.loads(urllib.request.urlopen(f"{SERVER}/history/{pid}").read())
    except: h={}
    if pid in h:
        for _,out in h[pid]["outputs"].items():
            for it in out.get("images",[]):
                q=urllib.parse.urlencode({"filename":it["filename"],"subfolder":it.get("subfolder",""),"type":"output"})
                data=urllib.request.urlopen(f"{SERVER}/view?{q}").read()
                dest=os.path.join(OUT,it["filename"]); open(dest,"wb").write(data); LAST=dest
                print(f"DONE in {time.time()-t0:.1f}s -> {dest} ({len(data)//1024} KB)")
        break
    time.sleep(1)
else: print("TIMEOUT")


In [ ]:
#@title 5 · Generate · Wan2.2 video { display-mode: "form" }
prompt = "A steampunk brass arrow slowly rotating on a wooden workshop base, dust motes drifting in warm window light, copper pipes in the background, cinematic, photorealistic" #@param {type:"string"}
negative = "色调艳丽，过曝，静态，细节模糊不清，字幕，静止，低质量，变形，水印" #@param {type:"string"}
seed = 7 #@param {type:"integer"}
width = 832 #@param {type:"integer"}
height = 480 #@param {type:"integer"}
frames = 33 #@param {type:"integer"}
steps = 30 #@param {type:"integer"}
import json, time, urllib.request, urllib.parse, os
SERVER = f"http://127.0.0.1:{PORT}"; OUT = f"{COMFY}/output"; os.makedirs(OUT, exist_ok=True)
wf = {"prompt":{
  "37":{"class_type":"UnetLoaderGGUF","inputs":{"unet_name":"wan2.2_ti2v_5B-Q4_K_M.gguf"}},
  "38":{"class_type":"CLIPLoader","inputs":{"clip_name":"umt5_xxl_fp8_e4m3fn_scaled.safetensors","type":"wan"}},
  "39":{"class_type":"VAELoader","inputs":{"vae_name":"wan2.2_vae.safetensors"}},
  "48":{"class_type":"ModelSamplingSD3","inputs":{"model":["37",0],"shift":8.0}},
  "6":{"class_type":"CLIPTextEncode","inputs":{"text":prompt,"clip":["38",0]}},
  "7":{"class_type":"CLIPTextEncode","inputs":{"text":negative,"clip":["38",0]}},
  "55":{"class_type":"Wan22ImageToVideoLatent","inputs":{"vae":["39",0],"width":width,"height":height,"length":frames,"batch_size":1}},
  "3":{"class_type":"KSampler","inputs":{"seed":seed,"steps":steps,"cfg":5.0,"sampler_name":"uni_pc","scheduler":"simple","denoise":1.0,"model":["48",0],"positive":["6",0],"negative":["7",0],"latent_image":["55",0]}},
  "8":{"class_type":"VAEDecode","inputs":{"samples":["3",0],"vae":["39",0]}},
  "47":{"class_type":"SaveWEBM","inputs":{"images":["8",0],"filename_prefix":"wan22","codec":"vp9","fps":16,"crf":20}}}}
pid = json.loads(urllib.request.urlopen(urllib.request.Request(f"{SERVER}/prompt",
    data=json.dumps(wf).encode(), headers={"Content-Type":"application/json"})).read())["prompt_id"]
print("submitted:", pid, f"(this can take several minutes — {frames} frames)")
t0=time.time(); LAST=None
while time.time()-t0<900:
    try: h=json.loads(urllib.request.urlopen(f"{SERVER}/history/{pid}").read())
    except: h={}
    if pid in h:
        for _,out in h[pid]["outputs"].items():
            for kind in ("gifs","images","videos"):
                for it in out.get(kind,[]):
                    q=urllib.parse.urlencode({"filename":it["filename"],"subfolder":it.get("subfolder",""),"type":it.get("type","output")})
                    data=urllib.request.urlopen(f"{SERVER}/view?{q}").read()
                    dest=os.path.join(OUT,it["filename"]); open(dest,"wb").write(data); LAST=dest
                    print(f"DONE in {time.time()-t0:.1f}s -> {dest} ({len(data)//1024} KB)")
        # also check for execution error
        if h[pid].get("status",{}).get("status_str")=="error":
            print("SERVER ERROR:", json.dumps(h[pid]["status"].get("messages",[]))[:500])
        break
    time.sleep(2)
else: print("TIMEOUT")


In [ ]:
#@title 6 · Display last result { display-mode: "form" }
from IPython.display import Image, Video, HTML, display
import os
if LAST:
    ext = os.path.splitext(LAST)[1].lower()
    if ext in (".webm",".mp4"):
        display(HTML(f'''<video controls src="data:video/webm;base64,{__import__('base64').b64encode(open(LAST,'rb').read()).decode()}"></video>'''))
    else:
        display(Image(filename=LAST))
else:
    print("nothing yet — run a Generate cell")
